# HarmShield — Hate and Offensive Content Detection

This notebook is a lightweight runner for the revised project. The task is multilabel NLP classification using HateXplain: one primary harmful-content label and five target-group labels. It does **not** claim to establish repeated cyberbullying behaviour from isolated posts.

## 1. Environment
Run the notebook from the project root. Install the pinned requirements if needed.

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / 'src').is_dir(), 'Open this notebook from the project root.'
print(PROJECT_ROOT)

In [ ]:
# Uncomment once in a fresh environment.
# %pip install -r requirements.txt

## 2. Dataset checks and exploratory analysis

In [ ]:
from src.data_loader import load_data
df = load_data()
print(df.shape)
df.head()

In [ ]:
%run run_eda.py

## 3. Leakage-safe model training
Each script reserves an untouched 20% final test set. Three-fold cross-validation and independent per-label OOF threshold selection operate only on the 80% development set using pooled out-of-fold probabilities.

In [ ]:
%run models/member1_logistic_regression.py
%run models/member2_svm.py
%run models/member3_random_forest.py

## 4. Final artifacts and results

In [ ]:
%run compare_models.py
%run generate_final_artifacts.py

In [ ]:
import pandas as pd
pd.read_csv('results/model_scores.csv')

## 5. Single-comment smoke test
The harmful verdict is controlled only by the primary label. Target-group outputs are contextual and cannot independently create a harmful verdict.

In [ ]:
from src.predictor import available_models, load_model, predict
model_name, model_path = next(iter(available_models().items()))
bundle = load_model(model_path)
thresholds = bundle.get('thresholds', {})
result = predict(bundle, 'I disagree with the proposal, but everyone deserves respect.')
print(model_name, thresholds, result['decision'], result['is_harmful'])
result['probs']